# RoadSense AI: Multi-Task Machine Learning for Accident Severity and Casualties
## Notebook 01: Data Ingestion, Verification, Cleaning & Feature Engineering
**Course Module**: Artificial Intelligence (CO5: Classification, Regression and Prediction Problems)
**Target Team Member**: Member 1 (Data Engineering & Preprocessing Lead)

---

### Objectives:
1. Ingest official UK STATS19 road safety data (Collisions and Vehicles for 2021, 2022, 2023).
2. Verify schema consistency and column alignment across all years.
3. Enforce **strict anti-leakage rules** (exclude crash dynamics, post-hoc injury adjustments, police response).
4. Apply **per-column unknown-code mapping** based on the official DfT Data Guide.
5. Engineer collision-level vehicle interaction features without arbitrary single-vehicle bias.
6. Define ground truth targets for Tasks A, B, and C.
7. Export the clean master dataset and data-quality audit report.

In [1]:
import os
import sys
import pandas as pd
import numpy as np

# Ensure project root is on sys.path
sys.path.append(os.path.abspath(".."))
from src.data import (
    UNKNOWN_CODE_MAP, 
    EXCLUDED_COLUMNS_RATIONALE, 
    verify_schema_consistency,
    build_master_dataset
)
print("Imports and configuration loaded successfully.")

Imports and configuration loaded successfully.


## 1. Schema Consistency Verification
Before concatenating datasets across multiple years, we verify that all 44 collision columns and 32 vehicle columns exist with identical naming and data types across 2021, 2022, and 2023.

In [2]:
schema_check = verify_schema_consistency(data_dir="../data", years=[2021, 2022, 2023])
print("All years schema-consistent:", schema_check["consistent"])
for yr, info in schema_check["collision_files"].items():
    print(f"Year {yr}: Collision cols = {info['col_count']}, Vehicle cols = {schema_check['vehicle_files'][yr]['col_count']}")

All years schema-consistent: True
Year 2021: Collision cols = 44, Vehicle cols = 32
Year 2022: Collision cols = 44, Vehicle cols = 32
Year 2023: Collision cols = 44, Vehicle cols = 32


## 2. Per-Column Unknown-Code Mapping
DfT STATS19 does not use a blanket missing value code. While `-1` generally indicates unrecorded data, several features use specialized codes (e.g. `99` for self-reported unknown speed limit or vehicle type, `9` for unknown weather/road surface, `7` for unknown lighting, `3` for unallocated urban/rural). We map each feature individually according to the DfT Data Guide.

In [3]:
for col, codes in UNKNOWN_CODE_MAP.items():
    print(f"Feature '{col:<25}' -> Unknown codes: {codes}")

Feature 'speed_limit              ' -> Unknown codes: [-1, 99]
Feature 'weather_conditions       ' -> Unknown codes: [-1, 9]
Feature 'road_type                ' -> Unknown codes: [-1, 9]
Feature 'light_conditions         ' -> Unknown codes: [-1, 7]
Feature 'junction_detail          ' -> Unknown codes: [-1, 99]
Feature 'road_surface_conditions  ' -> Unknown codes: [-1, 9]
Feature 'urban_or_rural_area      ' -> Unknown codes: [-1, 3]
Feature 'first_road_class         ' -> Unknown codes: [-1]
Feature 'junction_control         ' -> Unknown codes: [-1, 9]
Feature 'trunk_road_flag          ' -> Unknown codes: [-1]
Feature 'carriageway_hazards      ' -> Unknown codes: [-1, 99]
Feature 'vehicle_type             ' -> Unknown codes: [-1, 99]
Feature 'age_of_vehicle           ' -> Unknown codes: [-1, 99]
Feature 'age_band_of_driver       ' -> Unknown codes: [-1]
Feature 'police_force             ' -> Unknown codes: [-1, 99]


## 3. Strict Data Leakage Prevention Log
To prevent target leakage and unrealistic pre-crash model dependencies, we strictly exclude post-hoc outcomes and crash-dynamics fields.

In [4]:
leakage_df = pd.DataFrame([
    {"Dropped Column": col, "Leakage / Exclusion Rationale": reason}
    for col, reason in EXCLUDED_COLUMNS_RATIONALE.items()
])
print(f"Total explicitly excluded columns: {len(leakage_df)}")
leakage_df.head(10)

Total explicitly excluded columns: 45


,Dropped Column,Leakage / Exclusion Rationale
0,collision_injury_based,DfT post-accident classification: records whet...
1,collision_adjusted_severity_serious,DfT modeled probability: calculated post-hoc t...
2,collision_adjusted_severity_slight,DfT modeled probability: calculated post-hoc t...
3,enhanced_severity_collision,DfT post-accident administrative field: record...
4,first_point_of_impact,DfT crash dynamics field (code 1-4): records v...
5,skidding_and_overturning,DfT crash dynamics field (code 1-5): records w...
6,vehicle_leaving_carriageway,DfT crash dynamics field (code 1-8): records w...
7,hit_object_in_carriageway,DfT crash dynamics field (code 1-12): records ...
8,hit_object_off_carriageway,DfT crash dynamics field (code 1-11): records ...
9,vehicle_manoeuvre,DfT crash dynamics field (code 1-18): records ...


## 4. End-to-End Pipeline Execution
We execute the complete `build_master_dataset` pipeline, which:
- Aggregates vehicle records into collision-level interaction features (`any_motorcycle`, `any_pedal_cycle`, `any_hgv_or_bus`, `most_vulnerable_vehicle_type`, `age_of_vehicle_max`, `driver_age_band_youngest`).
- Cleans unknown codes per map.
- Extracts temporal markers (`hour`, `month`).
- Encodes targets for Tasks A (binary), B (3-class), and C (regression).
- Exports `data/clean_roadsense.parquet`.

In [5]:
master_df, dq_df = build_master_dataset(data_dir="../data", output_dir="../data", tables_dir="../outputs/tables")
print("Master clean shape:", master_df.shape)
master_df.head()

ROADSENSE AI: DATA PIPELINE INITIATING
[*] Schema consistency verified across years [2021, 2022, 2023]: True
[*] Processing Year 2021...


    -> Extracted 101,087 clean collision records.
[*] Processing Year 2022...


    -> Extracted 106,004 clean collision records.
[*] Processing Year 2023...


    -> Extracted 104,258 clean collision records.

[+] Master dataset aggregated: 311,349 total collision records.


[+] Saved clean master dataset to: ../data\clean_roadsense.parquet
[+] Saved 1,000-row sample to: ../data\clean_roadsense_sample.csv
[+] Data quality report saved to: ../outputs/tables\data_quality_report.csv
[+] Leakage exclusion log saved to: ../outputs/tables\leakage_exclusion_log.csv
Master clean shape: (311349, 29)


,collision_index,collision_year,police_force,target_severity_binary,target_severity_3class,target_casualties,speed_limit,weather_conditions,road_type,light_conditions,...,month,number_of_vehicles,any_motorcycle,any_pedal_cycle,any_hgv_or_bus,most_vulnerable_vehicle_type,age_of_vehicle_max,age_of_vehicle_max_missing,driver_age_band_youngest,driver_age_band_youngest_missing
0,2021170M10151,2021,17.0,0,2,1,30,1.0,6.0,1.0,...,2,1,0,0,0,car_taxi,NaN,1,9.0,0
1,2021170S11701,2021,17.0,1,1,1,30,1.0,3.0,4.0,...,10,2,0,1,0,pedal_cycle,NaN,1,3.0,0
2,2021111064046,2021,11.0,0,2,1,30,1.0,6.0,1.0,...,7,2,0,0,0,car_taxi,10.0,0,6.0,0
3,2021070535511,2021,7.0,0,2,1,60,1.0,6.0,1.0,...,8,2,0,0,0,car_taxi,3.0,0,5.0,0
4,2021070706357,2021,7.0,0,2,1,30,1.0,6.0,1.0,...,10,2,0,0,1,car_taxi,11.0,0,8.0,0


## 5. Data Quality Report & Missing Rates Audit
Examining missing rates and imputation strategies planned for `sklearn.compose.ColumnTransformer`.

In [6]:
dq_report = pd.read_csv("../outputs/tables/data_quality_report.csv")
dq_report.sort_values(by="missing_percent", ascending=False)

,feature,data_type,unique_values,missing_count,missing_percent,handled_unknown_codes,imputation_policy
10,junction_control,float64,4,136605,43.88,"[-1, 9]",Explicit 'Unknown' category
21,age_of_vehicle_max,float64,94,36747,11.80,NaN,Median imputation + Missing-indicator flag
11,trunk_road_flag,float64,2,22130,7.11,[-1],Explicit 'Unknown' category
23,driver_age_band_youngest,float64,11,15954,5.12,NaN,Median imputation + Missing-indicator flag
1,police_force,float64,43,12259,3.94,"[-1, 99]",Explicit 'Unknown' category
12,carriageway_hazards,float64,12,9939,3.19,"[-1, 99]",Explicit 'Unknown' category
3,weather_conditions,float64,8,9048,2.91,"[-1, 9]",Explicit 'Unknown' category
4,road_type,float64,5,8539,2.74,"[-1, 9]",Explicit 'Unknown' category
7,junction_detail,float64,6,6276,2.02,"[-1, 99]",Explicit 'Unknown' category
6,road_surface_conditions,float64,5,5898,1.89,"[-1, 9]",Explicit 'Unknown' category


## 6. Target Distributions & COVID Distribution Shift Note
### Target Balances:
- **Task A (Binary)**: Dangerous (~23.3%) vs Minor (~76.7%)
- **Task B (3-Class)**: Fatal (~1.48%), Serious (~21.8%), Slight (~76.7%)
- **Task C (Regression)**: Number of casualties (Mean 1.28, heavily right-skewed)

### 2021 COVID Distribution Shift:
> **Important**: The UK was subject to national COVID-19 lockdowns in early 2021, which resulted in reduced overall vehicular traffic but higher average speeds and a higher proportion of vulnerable road users (cyclists/pedestrians). Models trained on 2021–2022 and tested on 2023 capture this real-world macroeconomic and behavioral distribution shift.

In [7]:
for yr in [2021, 2022, 2023]:
    sub = master_df[master_df["collision_year"] == yr]
    print(f"Year {yr} ({len(sub):,} accidents):")
    print(f"  Dangerous rate: {sub['target_severity_binary'].mean()*100:.2f}%")
    print(f"  Fatal rate: {(sub['target_severity_3class'] == 0).mean()*100:.2f}%")
    print(f"  Avg casualties: {sub['target_casualties'].mean():.2f}")

Year 2021 (101,087 accidents):
  Dangerous rate: 22.51%
  Fatal rate: 1.46%
  Avg casualties: 1.27
Year 2022 (106,004 accidents):
  Dangerous rate: 23.52%
  Fatal rate: 1.51%
  Avg casualties: 1.28
Year 2023 (104,258 accidents):
  Dangerous rate: 23.94%
  Fatal rate: 1.46%
  Avg casualties: 1.28
